# Sentiment Panel Regression — Lagged Sentiment Control (t-2)

Robustness check: adds sentiment at t-2 as an additional control alongside the main t-1 sentiment variable.
All other specifications are identical to the baseline notebook.

In [1]:
import pandas as pd
import numpy as np
from linearmodels import PanelOLS, PooledOLS
from linearmodels.panel import RandomEffects
from pathlib import Path

## Config

| `LAG_WINDOW` | Sentiment window | Target return |
|---|---|---|
| `'daily'` | posts on day D-1 | return on day D |
| `'weekly'` | mean of posts in week W-1 | compound return in week W |
| `'monthly'` | mean of posts in month M-1 | compound return in month M |


In [2]:
LAG_WINDOW          = 'daily'   # 'daily' | 'weekly' | 'monthly'
NULL_STRATEGY       = 'drop'

PANEL_DATA_PATH  = Path('./data/panel_data.parquet')
SENTIMENT_PATH   = Path('../sentiment/data/all_labeled_finnishbert.parquet')
FORUM_POSTS_PATH = Path('../sentiment/data/cleaned_forum_posts.parquet')

print(f'LAG_WINDOW          : {LAG_WINDOW}')

LAG_WINDOW          : daily


## Load Data

In [3]:
stock_daily = pd.read_parquet(PANEL_DATA_PATH)
stock_daily['Date'] = pd.to_datetime(stock_daily['Date'])
if 'prev_day_sentiment' in stock_daily.columns:
    stock_daily = stock_daily.drop(columns=['prev_day_sentiment'])

sentiment   = pd.read_parquet(SENTIMENT_PATH,   columns=['id', 'sentiment'])
forum_posts = pd.read_parquet(FORUM_POSTS_PATH, columns=['id', 'date_time', 'ticker'])

print(f'Stock rows : {len(stock_daily):,}  |  tickers: {stock_daily["ticker"].nunique()}')
print(f'Sentiment  : {len(sentiment):,}')

Stock rows : 390,127  |  tickers: 162
Sentiment  : 532,424


## Resample Stock Data & Compute Controls

For weekly/monthly specs, daily returns are compounded into period returns and control variables are aggregated to the same frequency. Daily-lag controls (`return_lag1/2/5`) are replaced by a single `return_lag_period` computed from the resampled panel.

In [4]:
# Column classification for aggregation
COMPOUND_COLS = [
    'return',
    'open_to_close_return', 'close_to_open_return',
    'EURUSD_return', 'EURSEK_return', 'EURCNY_return',
    'GC_return', 'CL_return', 'VIX_return', 'GSPC_return', 'TNX_return',
    'OMXHPI_return', 'OMXN40_return', 'STOXX50E_return',
]
SUM_COLS  = ['euribor_3m_diff', 'finland_10y_diff']
MEAN_COLS = ['log_volume', 'gk_vol', 'momentum_12_1', 'amihud']
LAST_COLS = ['unemp_rate_change_lagged', 'cpi_rate_change_lagged', 'consumer_conf_change_lagged']

def compound(s):
    valid = s.dropna()
    return (1 + valid).prod() - 1 if len(valid) else np.nan


def resample_stock(df, period):
    """Resample daily panel to weekly or monthly frequency."""
    df = df.copy()
    if period == 'weekly':
        # Period key = Monday of the trading week
        df['_period'] = df['Date'] - pd.to_timedelta(df['Date'].dt.weekday, unit='D')
    else:  # monthly
        df['_period'] = df['Date'].dt.to_period('M').dt.to_timestamp()

    present = set(df.columns)
    agg = {}
    for c in COMPOUND_COLS:
        if c in present: agg[c] = compound
    for c in SUM_COLS:
        if c in present: agg[c] = 'sum'
    for c in MEAN_COLS:
        if c in present: agg[c] = 'mean'
    for c in LAST_COLS:
        if c in present: agg[c] = 'last'

    result = (
        df.groupby(['ticker', '_period'])
        .agg(agg)
        .reset_index()
        .rename(columns={'_period': 'Date'})
    )
    # Lagged period return replaces daily return_lag1/2/5
    result = result.sort_values(['ticker', 'Date'])
    result['return_lag_period'] = result.groupby('ticker')['return'].shift(1)
    return result


if LAG_WINDOW == 'daily':
    df_stock = stock_daily.copy()
    PERIOD_CONTROLS = [
        'return_lag1', 'return_lag2', 'return_lag5',
        'log_volume', 'gk_vol', 'momentum_12_1', 'amihud',
        'EURUSD_return', 'EURSEK_return', 'EURCNY_return',
        'GC_return', 'CL_return', 'VIX_return', 'GSPC_return', 'TNX_return',
        'euribor_3m_diff', 'OMXHPI_return', 'OMXN40_return', 'STOXX50E_return',
        'finland_10y_diff', 'unemp_rate_change_lagged',
        'cpi_rate_change_lagged', 'consumer_conf_change_lagged',
    ]
else:
    df_stock = resample_stock(stock_daily, LAG_WINDOW)
    PERIOD_CONTROLS = [
        'return_lag_period',
        'log_volume', 'gk_vol', 'momentum_12_1', 'amihud',
        'EURUSD_return', 'EURSEK_return', 'EURCNY_return',
        'GC_return', 'CL_return', 'VIX_return', 'GSPC_return', 'TNX_return',
        'euribor_3m_diff', 'OMXHPI_return', 'OMXN40_return', 'STOXX50E_return',
        'finland_10y_diff', 'unemp_rate_change_lagged',
        'cpi_rate_change_lagged', 'consumer_conf_change_lagged',
    ]

print(f'Resampled stock shape: {df_stock.shape}')
print(f'Period range: {df_stock["Date"].min().date()} — {df_stock["Date"].max().date()}')

Resampled stock shape: (390127, 28)
Period range: 2012-01-02 — 2025-12-30


## Compute Sentiment Variable

In [5]:
# Attach date + ticker to each label
sent_dated = sentiment.merge(forum_posts, on='id', how='inner')
sent_dated['Date'] = sent_dated['date_time'].dt.normalize()

# Daily mean sentiment per (ticker, calendar day)
daily_sent = (
    sent_dated
    .groupby(['ticker', 'Date'])['sentiment']
    .mean()
    .reset_index()
)

if LAG_WINDOW == 'daily':
    # Infer trading days from panel data: any weekday missing stock rows is
    # treated as a holiday (covers Easter, Independence Day, etc.).
    trading_days = np.sort(df_stock['Date'].unique())  # sorted datetime64 array

    # Map each post date to the next trading day *strictly after* it.
    # This absorbs weekend/holiday posts into the following trading day's
    # lag window rather than dropping them.
    #   e.g. Fri/Sat/Sun posts → all attributed to Monday's sentiment_var
    post_dates = daily_sent['Date'].values
    idx = np.searchsorted(trading_days, post_dates, side='right')
    valid = idx < len(trading_days)
    idx_clipped = np.minimum(idx, len(trading_days) - 1)
    daily_sent['next_trading_day'] = pd.to_datetime(
        np.where(valid, trading_days[idx_clipped], pd.NaT)
    )

    # Drop posts that fall after the last trading day in the dataset
    daily_sent = daily_sent.dropna(subset=['next_trading_day'])

    # Aggregate: all posts in the lag window [prev_trading_day+1 … D-1] → D
    lag_sent = (
        daily_sent
        .groupby(['ticker', 'next_trading_day'])['sentiment']
        .mean()
        .reset_index()
        .rename(columns={'next_trading_day': 'Date'})
    )

    # Merge onto trading-day grid; no shift needed — attribution is already lagged
    all_dates = df_stock[['ticker', 'Date']].drop_duplicates()
    daily_sent = all_dates.merge(lag_sent, on=['ticker', 'Date'], how='left')
    daily_sent['sentiment_var'] = daily_sent['sentiment']

    n_on_nontrading = (~np.isin(post_dates, trading_days) & valid).sum()
    print(f'Trading days inferred : {len(trading_days):,}')
    print(f'Posts on non-trading days absorbed: {n_on_nontrading:,}')

else:
    # Aggregate posts to the same period key used in df_stock
    if LAG_WINDOW == 'weekly':
        daily_sent['_period'] = daily_sent['Date'] - pd.to_timedelta(daily_sent['Date'].dt.weekday, unit='D')
    else:  # monthly
        daily_sent['_period'] = daily_sent['Date'].dt.to_period('M').dt.to_timestamp()

    period_sent = (
        daily_sent
        .groupby(['ticker', '_period'])['sentiment']
        .mean()
        .reset_index()
        .rename(columns={'_period': 'Date'})
    )

    # Merge onto period stock grid, then shift 1 period
    all_periods = df_stock[['ticker', 'Date']].drop_duplicates()
    period_sent = all_periods.merge(period_sent, on=['ticker', 'Date'], how='left')
    period_sent = period_sent.sort_values(['ticker', 'Date'])
    period_sent['sentiment_var'] = (
        period_sent
        .groupby('ticker')['sentiment']
        .transform(lambda s: s.shift(1))  # period P-1 sentiment predicts period P return
    )
    daily_sent = period_sent

# Apply null strategy
if NULL_STRATEGY == 'neutral':
    daily_sent['sentiment_var'] = daily_sent['sentiment_var'].fillna(1.0)
elif NULL_STRATEGY == 'forward_fill':
    daily_sent['sentiment_var'] = (
        daily_sent.groupby('ticker')['sentiment_var']
        .transform(lambda s: s.ffill())
    )
# 'drop' — leave NaN; handled by dropna() in panel prep

coverage = daily_sent['sentiment_var'].notna().mean()
print(f'Sentiment coverage after "{NULL_STRATEGY}" strategy: {coverage:.1%}')
print(daily_sent['sentiment_var'].describe())

Trading days inferred : 3,514
Posts on non-trading days absorbed: 15,543
Sentiment coverage after "drop" strategy: 18.4%
count    71732.000000
mean         1.212114
std          0.607126
min          0.000000
25%          0.916667
50%          1.181818
75%          1.750000
max          2.000000
Name: sentiment_var, dtype: float64


## Build Panel Dataset

In [6]:
df = df_stock.merge(
    daily_sent[['ticker', 'Date', 'sentiment_var']],
    on=['ticker', 'Date'], how='left'
)

# sentiment_var is already at t-1; shift one more period to get t-2
df = df.sort_values(['ticker', 'Date'])
df['sentiment_var_lag2'] = df.groupby('ticker')['sentiment_var'].shift(1)

analysis_vars = ['return', 'sentiment_var', 'sentiment_var_lag2'] + PERIOD_CONTROLS

df_panel = df.dropna(subset=analysis_vars)

df_panel = df_panel[['ticker', 'Date'] + analysis_vars].copy()
df_panel['const'] = 1.0  # explicit intercept for PooledOLS
df_panel = df_panel.set_index(['ticker', 'Date'])

print(f'Panel rows : {len(df_panel):,}')
print(f'Tickers    : {df_panel.index.get_level_values(0).nunique()}')
print(f'Periods    : {df_panel.index.get_level_values(1).nunique()}')

Panel rows : 38,022
Tickers    : 149
Periods    : 3069


## 1. Pooled OLS (Baseline)

## 1b. Hausman Test (FE vs RE)

In [7]:
exog_vars       = ['sentiment_var', 'sentiment_var_lag2'] + PERIOD_CONTROLS
exog_vars_const = ['const'] + exog_vars  # PooledOLS needs explicit intercept

re_results = RandomEffects(df_panel['return'], df_panel[exog_vars]).fit(
    cov_type='unadjusted'  # Hausman requires unadjusted (classical) SEs on both models
)
fe_hausman = PanelOLS(df_panel['return'], df_panel[exog_vars], entity_effects=True).fit(
    cov_type='unadjusted'
)

# Hausman statistic: H = (b_FE - b_RE)' * [Var(b_FE) - Var(b_RE)]^-1 * (b_FE - b_RE)
# Under H0 (RE consistent & efficient), H ~ chi2(k)
b_fe = fe_hausman.params.values
b_re = re_results.params[fe_hausman.params.index].values

v_fe = fe_hausman.cov.values
v_re = re_results.cov.loc[fe_hausman.params.index, fe_hausman.params.index].values

diff = b_fe - b_re
v_diff = v_fe - v_re

try:
    import numpy.linalg as la
    H = float(diff @ la.pinv(v_diff) @ diff)
    k = len(diff)
    from scipy import stats
    p_hausman = float(1 - stats.chi2.cdf(H, df=k))

    print('=' * 60)
    print('HAUSMAN TEST  (H0: RE is consistent — use RE)')
    print('=' * 60)
    print(f'Chi2 statistic : {H:.4f}')
    print(f'Degrees of freedom: {k}')
    print(f'p-value        : {p_hausman:.4f}')
    if p_hausman < 0.05:
        print('Verdict: Reject H0 → FE preferred (entity effects correlated with regressors)')
    else:
        print('Verdict: Fail to reject H0 → RE may be appropriate')
except np.linalg.LinAlgError as e:
    print(f'Hausman test failed (singular variance matrix): {e}')

print()
print('RE model summary (for reference):')
print(re_results.summary)

HAUSMAN TEST  (H0: RE is consistent — use RE)
Chi2 statistic : 374.3300
Degrees of freedom: 25
p-value        : 0.0000
Verdict: Reject H0 → FE preferred (entity effects correlated with regressors)

RE model summary (for reference):
                        RandomEffects Estimation Summary                        
Dep. Variable:                 return   R-squared:                        0.0984
Estimator:              RandomEffects   R-squared (Between):             -0.1116
No. Observations:               38022   R-squared (Within):               0.0998
Date:                Fri, May 01 2026   R-squared (Overall):              0.0984
Time:                        22:31:35   Log-likelihood                 7.868e+04
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      165.81
Entities:                         149   P-value                           0.0000
Avg Obs:                       255.18  

In [8]:
pooled_results = PooledOLS(df_panel['return'], df_panel[exog_vars_const]).fit(
    cov_type='clustered', cluster_entity=True
)
print('=' * 80)
print('POOLED OLS (Baseline — No Panel Effects)')
print('=' * 80)
print(pooled_results.summary)

POOLED OLS (Baseline — No Panel Effects)
                          PooledOLS Estimation Summary                          
Dep. Variable:                 return   R-squared:                        0.1011
Estimator:                  PooledOLS   R-squared (Between):             -0.0771
No. Observations:               38022   R-squared (Within):               0.1015
Date:                Fri, May 01 2026   R-squared (Overall):              0.1011
Time:                        22:31:35   Log-likelihood                 7.874e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      170.84
Entities:                         149   P-value                           0.0000
Avg Obs:                       255.18   Distribution:                F(25,37996)
Min Obs:                       1.0000                                           
Max Obs:                       1738.0   F-statistic (robust):       

## 2. Entity Fixed Effects

In [9]:
fe_results = PanelOLS(df_panel['return'], df_panel[exog_vars], entity_effects=True).fit(
    cov_type='clustered', cluster_entity=True
)
print('=' * 80)
print('ENTITY FIXED EFFECTS')
print('=' * 80)
print(fe_results.summary)

# Wooldridge-style test for serial autocorrelation in panel residuals
import statsmodels.formula.api as smf
resid = fe_results.resids.rename('resid')
resid_lag = resid.groupby(level='ticker').shift(1).rename('resid_lag')
resid_df = pd.concat([resid, resid_lag], axis=1).dropna().reset_index()
wr = smf.ols('resid ~ resid_lag', data=resid_df).fit(
    cov_type='cluster', cov_kwds={'groups': resid_df['ticker']}
)
print('\nWooldridge-style autocorrelation test (H0: no serial autocorrelation):')
print(f'  coef(resid_lag) = {wr.params["resid_lag"]:.4f},  p = {wr.pvalues["resid_lag"]:.4f}  {"→ autocorrelation present" if wr.pvalues["resid_lag"] < 0.05 else "→ no evidence of autocorrelation"}')


ENTITY FIXED EFFECTS
                          PanelOLS Estimation Summary                           
Dep. Variable:                 return   R-squared:                        0.1018
Estimator:                   PanelOLS   R-squared (Between):             -7.0471
No. Observations:               38022   R-squared (Within):               0.1018
Date:                Fri, May 01 2026   R-squared (Overall):             -0.0441
Time:                        22:31:35   Log-likelihood                 7.882e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      171.52
Entities:                         149   P-value                           0.0000
Avg Obs:                       255.18   Distribution:                F(25,37848)
Min Obs:                       1.0000                                           
Max Obs:                       1738.0   F-statistic (robust):             52.485
       

## 3. Two-Way Fixed Effects (Entity + Time)

In [10]:
fe_time_results = PanelOLS(
    df_panel['return'], df_panel[exog_vars],
    entity_effects=True, time_effects=True, drop_absorbed=True
).fit(cov_type='clustered', cluster_entity=True)

print('=' * 80)
print('TWO-WAY FIXED EFFECTS (Entity + Time FE)')
print('=' * 80)
print(fe_time_results.summary)

TWO-WAY FIXED EFFECTS (Entity + Time FE)
                          PanelOLS Estimation Summary                           
Dep. Variable:                 return   R-squared:                        0.0150
Estimator:                   PanelOLS   R-squared (Between):             -7.8690
No. Observations:               38022   R-squared (Within):               0.0064
Date:                Fri, May 01 2026   R-squared (Overall):             -0.1646
Time:                        22:31:35   Log-likelihood                 8.061e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      58.966
Entities:                         149   P-value                           0.0000
Avg Obs:                       255.18   Distribution:                 F(9,34796)
Min Obs:                       1.0000                                           
Max Obs:                       1738.0   F-statistic (robust):       

/var/folders/vs/mq6j80s94v7_ts2x082jdx1w0000gn/T/ipykernel_78234/3403796528.py:4: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

EURUSD_return, EURSEK_return, EURCNY_return, GC_return, CL_return, VIX_return, GSPC_return, TNX_return, euribor_3m_diff, OMXHPI_return, OMXN40_return, STOXX50E_return, finland_10y_diff, unemp_rate_change_lagged, cpi_rate_change_lagged, consumer_conf_change_lagged

  ).fit(cov_type='clustered', cluster_entity=True)


## 4. F-test for Entity Effects

In [11]:
from scipy import stats

# F-test for entity fixed effects (H0: Pooled OLS is adequate)
print('F-test for entity fixed effects:')
print('H0: Pooled OLS is appropriate (no entity effects)')
print('H1: Entity fixed effects are warranted')
try:
    f_stat = fe_results.f_pooled.stat
    f_pval = fe_results.f_pooled.pval
    print(f'  F-stat : {f_stat:.4f}')
    print(f'  p-value: {f_pval:.4f}')
    print('  Verdict:', 'Reject H0 — entity FE warranted' if f_pval < 0.05 else 'Fail to reject H0')
except Exception as e:
    print(f'  Could not compute F-test: {e}')

F-test for entity fixed effects:
H0: Pooled OLS is appropriate (no entity effects)
H1: Entity fixed effects are warranted
  F-stat : 1.1224
  p-value: 0.1472
  Verdict: Fail to reject H0


## 5. Sentiment Dummies (Entity FE, ref = neutral)

In [12]:
df_dum = df_panel.copy()
df_dum['sent_negative'] = (df_dum['sentiment_var'] < 1).astype(float)
df_dum['sent_positive'] = (df_dum['sentiment_var'] > 1).astype(float)

exog_dummies = ['sent_negative', 'sent_positive', 'sentiment_var_lag2'] + PERIOD_CONTROLS

fe_dummy_results = PanelOLS(df_dum['return'], df_dum[exog_dummies], entity_effects=True).fit(
    cov_type='clustered', cluster_entity=True
)
print('=' * 80)
print('ENTITY FE — SENTIMENT DUMMIES (ref: neutral)')
print('=' * 80)
print(fe_dummy_results.summary)

fe_time_dummy_results = PanelOLS(
    df_dum['return'], df_dum[exog_dummies],
    entity_effects=True, time_effects=True, drop_absorbed=True
).fit(cov_type='clustered', cluster_entity=True)
print('=' * 80)
print('TWO-WAY FE — SENTIMENT DUMMIES (ref: neutral)')
print('=' * 80)
print(fe_time_dummy_results.summary)


ENTITY FE — SENTIMENT DUMMIES (ref: neutral)
                          PanelOLS Estimation Summary                           
Dep. Variable:                 return   R-squared:                        0.1019
Estimator:                   PanelOLS   R-squared (Between):             -4.6467
No. Observations:               38022   R-squared (Within):               0.1019
Date:                Fri, May 01 2026   R-squared (Overall):              0.0062
Time:                        22:31:35   Log-likelihood                 7.883e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      165.22
Entities:                         149   P-value                           0.0000
Avg Obs:                       255.18   Distribution:                F(26,37847)
Min Obs:                       1.0000                                           
Max Obs:                       1738.0   F-statistic (robust):   

/var/folders/vs/mq6j80s94v7_ts2x082jdx1w0000gn/T/ipykernel_78234/2349295547.py:18: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

EURUSD_return, EURSEK_return, EURCNY_return, GC_return, CL_return, VIX_return, GSPC_return, TNX_return, euribor_3m_diff, OMXHPI_return, OMXN40_return, STOXX50E_return, finland_10y_diff, unemp_rate_change_lagged, cpi_rate_change_lagged, consumer_conf_change_lagged

  ).fit(cov_type='clustered', cluster_entity=True)


## 6. Piecewise Continuous Sentiment (Entity FE)

In [13]:
df_pw = df_panel.copy()
# Negative branch: intensity of negativity (0 when neutral/positive, up to 1 when fully negative)
df_pw['sent_neg_cont'] = np.where(df_pw['sentiment_var'] < 1, 1 - df_pw['sentiment_var'], 0.0)
# Positive branch: intensity of positivity (0 when neutral/negative, up to 1 when fully positive)
df_pw['sent_pos_cont'] = np.where(df_pw['sentiment_var'] > 1, df_pw['sentiment_var'] - 1, 0.0)

exog_pw = ['sent_neg_cont', 'sent_pos_cont', 'sentiment_var_lag2'] + PERIOD_CONTROLS

fe_pw_results = PanelOLS(df_pw['return'], df_pw[exog_pw], entity_effects=True).fit(
    cov_type='clustered', cluster_entity=True
)
print('=' * 80)
print('ENTITY FE — PIECEWISE CONTINUOUS SENTIMENT')
print('=' * 80)
print(fe_pw_results.summary)

fe_pw_time_results = PanelOLS(
    df_pw['return'], df_pw[exog_pw],
    entity_effects=True, time_effects=True, drop_absorbed=True
).fit(cov_type='clustered', cluster_entity=True)
print('=' * 80)
print('TWO-WAY FE — PIECEWISE CONTINUOUS SENTIMENT')
print('=' * 80)
print(fe_pw_time_results.summary)


ENTITY FE — PIECEWISE CONTINUOUS SENTIMENT
                          PanelOLS Estimation Summary                           
Dep. Variable:                 return   R-squared:                        0.1019
Estimator:                   PanelOLS   R-squared (Between):             -5.7949
No. Observations:               38022   R-squared (Within):               0.1019
Date:                Fri, May 01 2026   R-squared (Overall):             -0.0159
Time:                        22:31:36   Log-likelihood                 7.883e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      165.13
Entities:                         149   P-value                           0.0000
Avg Obs:                       255.18   Distribution:                F(26,37847)
Min Obs:                       1.0000                                           
Max Obs:                       1738.0   F-statistic (robust):     

/var/folders/vs/mq6j80s94v7_ts2x082jdx1w0000gn/T/ipykernel_78234/3898761333.py:20: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

EURUSD_return, EURSEK_return, EURCNY_return, GC_return, CL_return, VIX_return, GSPC_return, TNX_return, euribor_3m_diff, OMXHPI_return, OMXN40_return, STOXX50E_return, finland_10y_diff, unemp_rate_change_lagged, cpi_rate_change_lagged, consumer_conf_change_lagged

  ).fit(cov_type='clustered', cluster_entity=True)


## 7. Summary

In [14]:
print('=' * 80)
print('P-VALUE SUMMARY — ENTITY FE MODELS')
print('=' * 80)

def sig(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return 'n.s.'

# Continuous sentiment_var
rows = []
if 'sentiment_var' in fe_results.params.index:
    p    = fe_results.params['sentiment_var']
    se   = fe_results.std_errors['sentiment_var']
    pval = fe_results.pvalues['sentiment_var']
    rows.append({'Model': 'Entity FE', 'Coef': round(p, 6), 'Std Err': round(se, 6),
                 'p-value': round(pval, 4), 'Sig': sig(pval)})
else:
    rows.append({'Model': 'Entity FE', 'Coef': None, 'Std Err': None,
                 'p-value': None, 'Sig': 'n/a'})

print('Continuous sentiment_var:')
print(pd.DataFrame(rows).set_index('Model').to_string())

print('\nDummy model — Entity FE (ref=neutral):')
for var in ['sent_negative', 'sent_positive']:
    if var in fe_dummy_results.params.index:
        coef = fe_dummy_results.params[var]
        pval = fe_dummy_results.pvalues[var]
        print(f'  {var:15s}  coef={coef:.6f}  p={pval:.4f}  {sig(pval)}')
    else:
        print(f'  {var:15s}  n/a')

print('\nPiecewise continuous — Entity FE:')
for var in ['sent_neg_cont', 'sent_pos_cont']:
    if var in fe_pw_results.params.index:
        coef = fe_pw_results.params[var]
        pval = fe_pw_results.pvalues[var]
        print(f'  {var:15s}  coef={coef:.6f}  p={pval:.4f}  {sig(pval)}')
    else:
        print(f'  {var:15s}  n/a')

P-VALUE SUMMARY — ENTITY FE MODELS
Continuous sentiment_var:
              Coef   Std Err  p-value  Sig
Model                                     
Entity FE  0.00193  0.000344      0.0  ***

Dummy model — Entity FE (ref=neutral):
  sent_negative    coef=-0.001686  p=0.0011  ***
  sent_positive    coef=0.001051  p=0.0216  **

Piecewise continuous — Entity FE:
  sent_neg_cont    coef=-0.000702  p=0.2615  n.s.
  sent_pos_cont    coef=0.002728  p=0.0000  ***


## Residual Autocorrelation Diagnostics

ACF/PACF of Entity FE residuals (entity-averaged) to visually inspect for serial autocorrelation.

In [ ]:
from statsmodels.graphics import tsaplots
import matplotlib.pyplot as plt

def mean_resids(results):
    return results.resids.groupby(level='Date').mean().sort_index()

resids_fe   = mean_resids(fe_results)
resids_twfe = mean_resids(fe_time_results)

n_periods = len(resids_fe)
max_lags = min(20, n_periods // 2 - 1)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('ACF / PACF of Residuals (entity-averaged)', fontsize=13)

tsaplots.plot_acf( resids_fe,   lags=max_lags, ax=axes[0,0], alpha=0.05, zero=False)
tsaplots.plot_pacf(resids_fe,   lags=max_lags, ax=axes[0,1], alpha=0.05, zero=False, method='ywm')
tsaplots.plot_acf( resids_twfe, lags=max_lags, ax=axes[1,0], alpha=0.05, zero=False)
tsaplots.plot_pacf(resids_twfe, lags=max_lags, ax=axes[1,1], alpha=0.05, zero=False, method='ywm')

axes[0,0].set_title('Entity FE — ACF');   axes[0,0].set_xlabel('Lag (days)')
axes[0,1].set_title('Entity FE — PACF');  axes[0,1].set_xlabel('Lag (days)')
axes[1,0].set_title('Two-Way FE — ACF');  axes[1,0].set_xlabel('Lag (days)')
axes[1,1].set_title('Two-Way FE — PACF'); axes[1,1].set_xlabel('Lag (days)')

for ax in axes.flat:
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()


## Robustness: Driscoll-Kraay Standard Errors

Driscoll-Kraay SEs are robust to heteroskedasticity, serial autocorrelation, and cross-sectional dependence (common shocks across firms). Reported alongside clustered SEs for comparison.

In [16]:
import numpy as np
import pandas as pd

T = df_panel.index.get_level_values('Date').nunique()
bw = int(np.floor(T ** 0.25))
print(f'Time periods T={T}, Driscoll-Kraay bandwidth={bw}')

def fit_dk(df, dv, exog, entity_only=False):
    kwargs = dict(entity_effects=True)
    if not entity_only:
        kwargs.update(time_effects=True, drop_absorbed=True)
    return PanelOLS(df[dv], df[exog], **kwargs).fit(cov_type='driscoll-kraay', bandwidth=bw)

# Entity FE
fe_dk       = fit_dk(df_panel, 'return', exog_vars,    entity_only=True)
fe_dum_dk   = fit_dk(df_dum,   'return', exog_dummies, entity_only=True)
fe_pw_dk    = fit_dk(df_pw,    'return', exog_pw,      entity_only=True)

# TWFE
twfe_dk     = fit_dk(df_panel, 'return', exog_vars)
twfe_dum_dk = fit_dk(df_dum,   'return', exog_dummies)
twfe_pw_dk  = fit_dk(df_pw,    'return', exog_pw)

def compare_row(model, spec, var, clustered, dk):
    if var not in clustered.params.index:
        return None
    return {
        'Model': model, 'Spec': spec, 'Variable': var,
        'Coef': round(clustered.params[var], 6),
        'SE (clustered)': round(clustered.std_errors[var], 6),
        'p (clustered)': round(clustered.pvalues[var], 6),
        'SE (DK)': round(dk.std_errors[var], 6),
        'p (DK)': round(dk.pvalues[var], 6),
    }

rows = [
    compare_row('Entity FE',  'Continuous', 'sentiment_var',  fe_results,            fe_dk),
    compare_row('Two-Way FE', 'Continuous', 'sentiment_var',  fe_time_results,       twfe_dk),
    compare_row('Entity FE',  'Dummies',    'sent_negative',  fe_dummy_results,      fe_dum_dk),
    compare_row('Two-Way FE', 'Dummies',    'sent_negative',  fe_time_dummy_results, twfe_dum_dk),
    compare_row('Entity FE',  'Dummies',    'sent_positive',  fe_dummy_results,      fe_dum_dk),
    compare_row('Two-Way FE', 'Dummies',    'sent_positive',  fe_time_dummy_results, twfe_dum_dk),
    compare_row('Entity FE',  'Piecewise',  'sent_neg_cont',  fe_pw_results,         fe_pw_dk),
    compare_row('Two-Way FE', 'Piecewise',  'sent_neg_cont',  fe_pw_time_results,    twfe_pw_dk),
    compare_row('Entity FE',  'Piecewise',  'sent_pos_cont',  fe_pw_results,         fe_pw_dk),
    compare_row('Two-Way FE', 'Piecewise',  'sent_pos_cont',  fe_pw_time_results,    twfe_pw_dk),
]
rows = [r for r in rows if r is not None]

print('\nClustered vs Driscoll-Kraay SE comparison (Entity FE and TWFE):')
print(pd.DataFrame(rows).to_string(index=False))


Time periods T=3069, Driscoll-Kraay bandwidth=7


/var/folders/vs/mq6j80s94v7_ts2x082jdx1w0000gn/T/ipykernel_78234/1636901671.py:12: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

EURUSD_return, EURSEK_return, EURCNY_return, GC_return, CL_return, VIX_return, GSPC_return, TNX_return, euribor_3m_diff, OMXHPI_return, OMXN40_return, STOXX50E_return, finland_10y_diff, unemp_rate_change_lagged, cpi_rate_change_lagged, consumer_conf_change_lagged

  return PanelOLS(df[dv], df[exog], **kwargs).fit(cov_type='driscoll-kraay', bandwidth=bw)
/var/folders/vs/mq6j80s94v7_ts2x082jdx1w0000gn/T/ipykernel_78234/1636901671.py:12: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

EURUSD_return, EURSEK_return, EURCNY_return, GC_return, CL_return, VIX_return, GSPC_return, TNX_return, euribor_3m_diff, OMXHPI_return, OMXN40_return, STOXX50E_return, finland_10y_diff, unemp_rate_change_lagged, cpi_rate_change_lagged, consumer_conf_change_lagged

  return PanelO


Clustered vs Driscoll-Kraay SE comparison (Entity FE and TWFE):
     Model       Spec      Variable      Coef  SE (clustered)  p (clustered)  SE (DK)   p (DK)
 Entity FE Continuous sentiment_var  0.001930        0.000344       0.000000 0.000326 0.000000
Two-Way FE Continuous sentiment_var  0.001900        0.000359       0.000000 0.000333 0.000000
 Entity FE    Dummies sent_negative -0.001686        0.000515       0.001060 0.000437 0.000112
Two-Way FE    Dummies sent_negative -0.001627        0.000558       0.003541 0.000478 0.000664
 Entity FE    Dummies sent_positive  0.001051        0.000458       0.021638 0.000421 0.012474
Two-Way FE    Dummies sent_positive  0.001086        0.000477       0.022726 0.000441 0.013784
 Entity FE  Piecewise sent_neg_cont -0.000702        0.000626       0.261532 0.000545 0.197670
Two-Way FE  Piecewise sent_neg_cont -0.000703        0.000679       0.300417 0.000587 0.230896
 Entity FE  Piecewise sent_pos_cont  0.002728        0.000512       0.000000 0.0

/var/folders/vs/mq6j80s94v7_ts2x082jdx1w0000gn/T/ipykernel_78234/1636901671.py:12: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

EURUSD_return, EURSEK_return, EURCNY_return, GC_return, CL_return, VIX_return, GSPC_return, TNX_return, euribor_3m_diff, OMXHPI_return, OMXN40_return, STOXX50E_return, finland_10y_diff, unemp_rate_change_lagged, cpi_rate_change_lagged, consumer_conf_change_lagged

  return PanelOLS(df[dv], df[exog], **kwargs).fit(cov_type='driscoll-kraay', bandwidth=bw)
